[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C52_Industrial_Research_Practice_Course/00_setup/00_environment_check.ipynb)

# 00 · 环境自检与「三条纪律」热身

本课全程 **纯 numpy + 标准库、CPU、不联网**。用 numpy 复现框架/运行时/硬件的**语义**——
因为这些工具的困难几乎都不在「怎么调 API」，而在「它的语义与你以为的不一样」。

这个 notebook 做三件事：① 环境自检；② 用三个最小例子体会本课的三条纪律
（**数值对拍 / 版本即环境 / 边界要显式**）；③ 建立贯穿全课的工具函数。

## 1 · 环境自检

In [ ]:
import sys, platform, math, json, itertools, collections
print('Python', sys.version.split()[0], '|', platform.system(), platform.machine())
import numpy as np; print('numpy', np.__version__)
for name in ['torch', 'tensorflow', 'onnx', 'onnxruntime']:
    try:
        m = __import__(name); print(f'  {name:<12s} {getattr(m, "__version__", "?")} (有则更好)')
    except ImportError:
        print(f'  {name:<12s} 未安装 -> 走 numpy 复现路径')
print('\n环境就绪 ✅  —— 本课不需要 TensorFlow / ONNX / CUDA / 联网')

## 2 · 纪律一：数值对拍 —— 「跑通了」不等于「对了」

跨框架迁移、图导出、量化优化，三者的共同失败模式是：**代码不报错、输出形状也对，但数值错了**。
所以本课的每一次「变换」都要给出**逐层的数值等价证明**。

先把这套方法学封装成可复用的函数。

In [ ]:
def allclose_report(a, b, rtol=1e-5, atol=1e-6, name=''):
    '''逐张量对拍：不只给 True/False，还给出**最大绝对/相对误差与位置** ——
       因为「差在哪里」比「差不差」更有诊断价值。'''
    a, b = np.asarray(a, dtype=np.float64), np.asarray(b, dtype=np.float64)
    if a.shape != b.shape:
        return {'name': name, 'ok': False, 'reason': f'shape {a.shape} vs {b.shape}'}
    diff = np.abs(a - b)
    denom = np.maximum(np.abs(b), 1e-12)
    rel = diff / denom
    idx = int(np.argmax(diff))
    return {'name': name, 'ok': bool(np.allclose(a, b, rtol=rtol, atol=atol)),
            'max_abs': float(diff.max()), 'max_rel': float(rel.max()),
            'at': np.unravel_index(idx, a.shape), 'mean_abs': float(diff.mean())}

def layerwise_check(ref_acts, new_acts, rtol=1e-5, atol=1e-6):
    '''**逐层**对拍：只比最终输出会掩盖「前面错了、后面碰巧抵消」的情况。
       返回第一个不匹配的层 —— 这才是定位问题的正确方式。'''
    rows = []
    for k in ref_acts:
        r = allclose_report(ref_acts[k], new_acts.get(k, np.array([])), rtol, atol, name=k)
        rows.append(r)
        if not r['ok']:
            break                      # 第一个出错的层就是问题所在，后面的都是它的后果
    return rows

rng = np.random.default_rng(0)
x = rng.normal(size=(4, 8))
acts_ref = {'l1': x @ np.ones((8, 8)), 'l2': np.tanh(x @ np.ones((8, 8)))}
acts_ok = {'l1': acts_ref['l1'].copy(), 'l2': acts_ref['l2'].copy()}
acts_bad = {'l1': acts_ref['l1'] + 1e-3, 'l2': acts_ref['l2'].copy()}

for name, acts in [('完全一致', acts_ok), ('第一层就错', acts_bad)]:
    rows = layerwise_check(acts_ref, acts)
    status = '✅ 全部通过' if all(r['ok'] for r in rows) else f'❌ 首个失配: {rows[-1]["name"]}'
    print(f'{name:<12s} {status}  (检查了 {len(rows)} 层)')
    if not rows[-1]['ok']:
        print(f'    max_abs={rows[-1]["max_abs"]:.2e} at {rows[-1]["at"]}')

assert all(r['ok'] for r in layerwise_check(acts_ref, acts_ok))
assert not layerwise_check(acts_ref, acts_bad)[-1]['ok']
print('\n✅ 逐层对拍就位。**只比最终输出**会掩盖「前面错了后面抵消」的情况 ——')
print('   而那种情况在换了输入分布之后就会暴露（且极难定位）。')

### 容差怎么定：不同精度的合理阈值

「对拍失败」常常只是容差设错了。**不同 dtype 的机器精度差三个数量级。**

In [ ]:
EPS = {'float64': 2.2e-16, 'float32': 1.2e-7, 'float16': 9.8e-4, 'bfloat16': 7.8e-3}

def suggest_tolerance(dtype, depth, safety=10.0):
    '''经验法则：误差随层数近似按 sqrt(depth) 累积（随机游走），再留一个安全系数。'''
    eps = EPS[dtype]
    return {'rtol': safety * eps * math.sqrt(max(1, depth)),
            'atol': safety * eps * math.sqrt(max(1, depth))}

print(f"{'dtype':<10s} {'1 层':>12s} {'12 层':>12s} {'96 层':>12s}")
for dt in EPS:
    ts = [suggest_tolerance(dt, d)['rtol'] for d in (1, 12, 96)]
    print(f'{dt:<10s} {ts[0]:>12.2e} {ts[1]:>12.2e} {ts[2]:>12.2e}')

t32 = suggest_tolerance('float32', 12)['rtol']
t16 = suggest_tolerance('float16', 12)['rtol']
assert t16 > t32 * 100, 'fp16 的合理容差比 fp32 大两个数量级以上'
assert suggest_tolerance('float32', 96)['rtol'] > suggest_tolerance('float32', 1)['rtol']
print('\n✅ 两条推论：')
print('   ① 用 fp32 的容差去对拍 fp16 的导出结果，必然「失败」—— 但那不是 bug。')
print('   ② 深层模型的累积误差更大 —— 所以**逐层对拍**比只看最终输出更可靠：')
print('      前几层的容差可以很紧，问题会在它真正发生的那一层暴露。')

## 3 · 纪律二：版本即环境

跨框架 / 跨运行时的问题里，很大一部分是**版本问题**：某个算子在 opset 13 才支持、
CUDA runtime 必须 ≤ driver 支持的版本、某个 API 在 2.x 改了默认值。

**凡是报告一个问题，必须同时报告版本三元组。**

In [ ]:
def env_fingerprint(**versions):
    '''把环境压成一个可比较、可写进日志的指纹。'''
    items = sorted(versions.items())
    s = ';'.join(f'{k}={v}' for k, v in items)
    return {'string': s, 'dict': dict(items)}

def diff_env(a, b):
    '''两个环境的差异 —— 排查「在我这好使」时的第一步。'''
    keys = sorted(set(a['dict']) | set(b['dict']))
    return [(k, a['dict'].get(k, '—'), b['dict'].get(k, '—'))
            for k in keys if a['dict'].get(k) != b['dict'].get(k)]

dev = env_fingerprint(python='3.11', torch='2.4.1', onnx='1.16.0', opset='17',
                      cuda_runtime='12.1', driver='550.54')
prod = env_fingerprint(python='3.11', torch='2.4.1', onnx='1.14.0', opset='14',
                       cuda_runtime='12.1', driver='525.85')
print('开发环境:', dev['string'])
print('生产环境:', prod['string'])
print('\n差异:')
for k, a, b in diff_env(dev, prod):
    print(f'  {k:<14s} dev={a:<10s} prod={b}')

d = {k: (a, b) for k, a, b in diff_env(dev, prod)}
assert 'opset' in d and 'driver' in d
assert not diff_env(dev, dev), '同一环境不应有差异'
print('\n✅ 这三行差异就是「在我这好使」类问题的头号嫌疑人：')
print('   · opset 17 -> 14：某些算子在低 opset 不存在（模块 03）')
print('   · driver 550 -> 525：CUDA runtime 12.1 可能超出 525 的支持上限（模块 04）')

## 4 · 纪律三：边界要显式

交出去的模型必须附带「它在什么条件下成立」。**不写清边界，等于把问题推给下游。**

In [ ]:
class Contract:
    '''交付契约：把「这个模型在什么条件下有效」写成可检查的对象。'''
    def __init__(self, name, input_shapes, dtype, opset=None, max_batch=None,
                 max_seq=None, tolerance=None, notes=()):
        self.name, self.input_shapes, self.dtype = name, input_shapes, dtype
        self.opset, self.max_batch, self.max_seq = opset, max_batch, max_seq
        self.tolerance, self.notes = tolerance or {}, list(notes)

    def validate(self, batch, seq, dtype):
        errs = []
        if self.max_batch is not None and batch > self.max_batch:
            errs.append(f'batch {batch} > 支持上限 {self.max_batch}')
        if self.max_seq is not None and seq > self.max_seq:
            errs.append(f'seq_len {seq} > 支持上限 {self.max_seq}')
        if dtype != self.dtype:
            errs.append(f'dtype {dtype} != 导出时的 {self.dtype}')
        return (not errs), errs

    def render(self):
        lines = [f'# 交付契约: {self.name}', '',
                 f'- 输入形状: {self.input_shapes}',
                 f'- dtype: {self.dtype}',
                 f'- opset: {self.opset}',
                 f'- 支持的最大 batch: {self.max_batch}',
                 f'- 支持的最大序列长: {self.max_seq}',
                 f'- 与参考实现的容差: {self.tolerance}']
        if self.notes:
            lines += ['- 已知限制:'] + [f'    · {n}' for n in self.notes]
        return '\n'.join(lines)

c = Contract('text-classifier-v3',
             input_shapes={'input_ids': ['batch', 'seq'], 'attention_mask': ['batch', 'seq']},
             dtype='float16', opset=17, max_batch=64, max_seq=512,
             tolerance={'rtol': 1e-2, 'atol': 1e-2},
             notes=['seq_len 必须是 8 的倍数（TensorRT 的对齐要求）',
                    'batch=1 时走另一条 kernel，延迟不成比例',
                    'fp16 下极端输入（|x|>1e4）可能溢出'])
print(c.render())

ok, errs = c.validate(batch=128, seq=1024, dtype='float32')
print(f'\n用 batch=128, seq=1024, fp32 调用 -> 合法? {ok}')
for e in errs: print(f'  ❌ {e}')
assert not ok and len(errs) == 3
ok2, _ = c.validate(batch=32, seq=256, dtype='float16')
assert ok2
print('\n✅ 把「适用范围」写成可检查的对象，而不是 README 里的一段话 ——')
print('   前者会在越界时报错，后者只会在事故复盘时被翻出来。')

## 5 · ✏️ 练习：环境兼容性判断

实现 `cuda_compatible(driver_version, cuda_runtime)`：NVIDIA 的规则是
**driver 必须 ≥ 对应 CUDA runtime 的最低 driver 版本**（向后兼容，但不向前）。
用下表判断，返回 `(是否兼容, 说明字符串)`。

| CUDA runtime | 最低 driver |
|---|---|
| 11.8 | 520.61 |
| 12.1 | 530.30 |
| 12.4 | 550.54 |

In [ ]:
MIN_DRIVER = {'11.8': 520.61, '12.1': 530.30, '12.4': 550.54}

def cuda_compatible(driver_version, cuda_runtime):
    # TODO: 查表得最低 driver；driver_version >= 它 -> (True, '兼容')
    #       否则 (False, f'需要 driver >= X，当前 Y')
    #       未知的 cuda_runtime -> (False, 'unknown CUDA runtime ...')
    raise NotImplementedError

In [ ]:
# —— 练习自测 ——
ok, msg = cuda_compatible(550.54, '12.1')
assert ok, msg
ok2, msg2 = cuda_compatible(525.85, '12.1')
assert not ok2 and '530.3' in msg2, msg2
assert cuda_compatible(520.61, '11.8')[0], '刚好等于最低版本应兼容'
assert not cuda_compatible(510.0, '11.8')[0]
assert not cuda_compatible(999.0, '13.0')[0], '未知 runtime 应保守判为不兼容'
for drv, rt in [(550.54, '12.4'), (530.30, '12.4'), (525.85, '12.1')]:
    ok_, m_ = cuda_compatible(drv, rt)
    print(f'driver {drv} + CUDA {rt} -> {"✅" if ok_ else "❌ " + m_}')
print('\n✅ 练习通过：这就是「镜像里的 CUDA 版本必须 ≤ 宿主 driver 支持的版本」')
print('   （C48 模块 01 提过这个坑，模块 04 会把整张矩阵讲清楚）')

---
### 📖 参考答案

In [ ]:
def cuda_compatible(driver_version, cuda_runtime):
    if cuda_runtime not in MIN_DRIVER:
        return False, f'unknown CUDA runtime {cuda_runtime}'
    need = MIN_DRIVER[cuda_runtime]
    if driver_version >= need:
        return True, '兼容'
    return False, f'需要 driver >= {need}，当前 {driver_version}'

## 6 · 🧪 胶囊：五个模块各自的「交付物」

预告本课每个模块的产出。它们都是可以直接放进你项目工具库的东西。

In [ ]:
DELIVERABLES = [
    ('模块 01', 'tf.function 语义复现器', '理解重追踪与副作用陷阱；读懂 TF 代码'),
    ('模块 02', '权重映射表 + 逐层对拍脚本', '把任意框架的权重搬过去并**证明**等价'),
    ('模块 03', '导出前检查器（算子覆盖 + 动态轴 + 契约）', '避免「导出成功但线上形状一变就崩」'),
    ('模块 04', '显存分解器 + OOM 诊断树 + 版本矩阵检查', '把「跑不起来」变成「缺 X GB，因为 Y」'),
    ('模块 05', '可专利性自检 + 交底书模板 + 许可兼容矩阵', '识别值得申请的想法；避免许可地雷'),
]
print(f"{'模块':<8s} {'交付物':<34s} {'解决什么'}")
for m, d, w in DELIVERABLES:
    print(f'{m:<8s} {d:<34s} {w}')
assert len(DELIVERABLES) == 5
print('\n✅ 五个交付物都是**可以直接放进项目工具库**的东西 ——')
print('   这门课的价值不在「知道有这回事」，而在「下次遇到时有现成的工具」。')

✅ 检查全部通过即环境就绪、方法论到位。

**本课的契约**：每个模块都会 ① 用 numpy **复现**目标工具的关键语义（带 assert），
② 给出**可原样使用**的真实命令/代码/模板，③ 产出一个**可复用的检查器或清单**。

**接下来五个模块**：01 TensorFlow/Keras 心智模型 → 02 框架迁移与权重对齐 →
03 模型导出与推理运行时 → 04 真机 GPU 工作流 → 05 研究产出与知识产权。

**它们彼此独立，可以按需单独读。** 下一站：**模块 01 · TensorFlow / Keras 心智模型**。